# JED Attack Submission

Run all cells → **Save Version** → **Submit to Competition**.

Inputs needed:
- competition data (already attached)
- dataset `poojankumartandel/jed-attack-submission`

In [ ]:
import shutil
import zipfile
from pathlib import Path

WORKING = Path("/kaggle/working")
INPUT = Path("/kaggle/input")

print("=== dataset / competition roots ===")
for p in sorted(INPUT.glob("*")):
    print(" ", p)


def _extract_zip(zpath: Path, dest: Path) -> Path | None:
    if dest.exists():
        shutil.rmtree(dest)
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zpath, "r") as zf:
        zf.extractall(dest)
    hits = list(dest.rglob("attack.py"))
    return hits[0].parent if hits else None


def find_attack_root() -> Path:
    # Prefer the attached dataset (never prefer /kaggle/working first).
    preferred = [
        INPUT / "datasets" / "poojankumartandel" / "jed-attack-submission",
        INPUT / "jed-attack-submission",
    ]
    for root in preferred:
        if (root / "attack.py").exists():
            return root

    hits = [p for p in INPUT.rglob("attack.py") if "competitions" not in p.parts]
    if hits:
        return hits[0].parent

    for zpath in INPUT.rglob("*.zip"):
        if "competitions" in zpath.parts:
            continue
        print("Trying zip:", zpath)
        root = _extract_zip(zpath, WORKING / "_extracted_submission")
        if root is not None:
            return root

    raise FileNotFoundError(
        "Attach dataset poojankumartandel/jed-attack-submission via Add Input, then re-run."
    )


def install_framework(src: Path) -> None:
    dst_attack = WORKING / "attack.py"
    dst_af = WORKING / "attack_framework"

    src_attack = src / "attack.py"
    if src_attack.resolve() != dst_attack.resolve():
        shutil.copy2(src_attack, dst_attack)

    if dst_af.exists():
        shutil.rmtree(dst_af)

    af_src = src / "attack_framework"
    if af_src.is_dir():
        if af_src.resolve() == dst_af.resolve():
            return
        shutil.copytree(af_src, dst_af)
        return

    af_zip = src / "attack_framework.zip"
    if af_zip.is_file():
        dst_af.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(af_zip, "r") as zf:
            zf.extractall(dst_af)
        nested = dst_af / "attack_framework"
        if nested.is_dir() and not (dst_af / "__init__.py").exists():
            for item in nested.iterdir():
                shutil.move(str(item), dst_af / item.name)
            shutil.rmtree(nested)
        return

    raise FileNotFoundError(f"attack_framework missing next to {src}")


src = find_attack_root()
print("Source root:", src)
install_framework(src)
print("Ready attack.py:", (WORKING / "attack.py").exists())
print("Ready framework:", sorted(p.name for p in (WORKING / "attack_framework").glob("*.py")))

In [ ]:
import csv
import glob
import importlib.util
import os
import py_compile
import sys
from pathlib import Path

IS_RERUN = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

# Competition data path on current Kaggle layout
candidates = [
    "/kaggle/input/competitions/ai-agent-security-multi-step-tool-attacks",
    "/kaggle/input/ai-agent-security-multi-step-tool-attacks",
    *glob.glob("/kaggle/input/**/kaggle_evaluation", recursive=True),
]
sdk_root = None
for p in candidates:
    root = p
    if p.endswith("kaggle_evaluation"):
        root = str(Path(p).parent)
    if os.path.isdir(os.path.join(root, "kaggle_evaluation")):
        sdk_root = root
        break

if sdk_root is None:
    raise FileNotFoundError("Competition data with kaggle_evaluation not attached")
if sdk_root not in sys.path:
    sys.path.insert(0, sdk_root)
print("SDK path:", sdk_root)

ATTACK_PATH = "/kaggle/working/attack.py"
assert Path(ATTACK_PATH).exists(), "Run previous cell first"
py_compile.compile(ATTACK_PATH, doraise=True)
if "/kaggle/working" not in sys.path:
    sys.path.insert(0, "/kaggle/working")

spec = importlib.util.spec_from_file_location("attack_candidate", ATTACK_PATH)
module = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(module)

smoke = module.AttackAlgorithm().run(None, None)
assert smoke
assert all(1 <= len(c.user_messages) <= 32 for c in smoke)
assert all(isinstance(m, str) and 0 < len(m) <= 2000 for c in smoke for m in c.user_messages)
print("attack.py compile/import/structure smoke: PASS")

from kaggle_evaluation.jed_attack_134815.jed_attack_inference_server import JEDAttackInferenceServer

server = JEDAttackInferenceServer()
if IS_RERUN:
    server.serve()
else:
    with open("/kaggle/working/submission.csv", "w", newline="") as fh:
        writer = csv.writer(fh)
        writer.writerow(["Id", "Score"])
        writer.writerows([
            ["gpt_oss_public", 0.0],
            ["gpt_oss_private", 0.0],
            ["gemma_public", 0.0],
            ["gemma_private", 0.0],
        ])
    print("placeholder submission.csv written (not a competition rerun)")
    print("For a real score: Save Version → Submit to Competition")